# 02 · Extract V-JEPA 2.1 features for the 25-game set
**GPU job — needs a T4.** Reads videos from `soccernet-25games`, writes `(T,1024)` `*_VJEPA21_L.npy` per half, saves them as dataset `vjepa-features-25games`.

**Resumable:** extraction runs in chunks and saves the feature dataset after each chunk. If the session drops, attach the partial `vjepa-features-25games` + re-run — done halves are skipped.

**Kaggle settings before running:**
- Accelerator: **GPU T4 x2** (P100 is incompatible)
- Internet: **ON** (torch.hub model download, ~4.8 GB, once)
- Add data: **`daniels02/soccernet-25games`** (videos)
- Add data (optional, speeds resume): **`daniels02/vjepa-features-3games`** and/or a partial `vjepa-features-25games` from a previous session
- No secret needed (no download from SoccerNet here)

## Guard cell — require a T4 GPU + video dataset attached

In [ ]:
import torch, glob
from pathlib import Path

assert torch.cuda.is_available(), "No GPU. Set Accelerator = GPU T4 x2."
name = torch.cuda.get_device_name(0)
print("GPU:", name)
assert "T4" in name, f"Got {name}; this needs a T4 (P100 sm_60 is incompatible). Switch accelerator."

vids = glob.glob("/kaggle/input/**/*_224p.mkv", recursive=True)
print("videos found under /kaggle/input:", len(vids), "(expect 50 from soccernet-25games)")
assert vids, "No videos. Attach the soccernet-25games dataset."


## Clone repo + install deps

In [ ]:
import os
REPO = "https://github.com/DanielSch02/Football_JEPA.git"
%cd /kaggle/working
!rm -rf Football_JEPA
!git clone {REPO}
%cd Football_JEPA
!git log --oneline -1
!pip install -q SoccerNet torchcodec decord


## Set up working dirs + seed any already-extracted features
Features are written to `/kaggle/working/soccernet`. We seed it with any existing `*_VJEPA21_L.npy` from attached feature datasets so those halves are skipped (resume).

In [ ]:
import shutil, glob
from pathlib import Path
from footy.config import VJEPA_GAME_PATHS_FULL, default_data_dir

VIDEO_DIR = default_data_dir()                 # auto-detected soccernet-25games mount
WORK = "/kaggle/working/soccernet"
print("VIDEO_DIR:", VIDEO_DIR)

# seed previously-extracted features (from vjepa-features-3games or a partial 25 run)
seeded = 0
for npy in glob.glob("/kaggle/input/**/*_VJEPA21_L.npy", recursive=True):
    p = Path(npy)
    # path tail: england_epl/<season>/<game>/<half>_VJEPA21_L.npy
    rel = Path(*p.parts[-4:])
    dst = Path(WORK) / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy(npy, dst); seeded += 1
print(f"seeded {seeded} existing feature files (will be skipped)")
print("features present so far:", len(glob.glob(f"{WORK}/**/*_VJEPA21_L.npy", recursive=True)), "/ 50")


## Extract — chunk 1 of 5 (games 1-5)
Each chunk extracts ~5 games then saves the feature dataset. `extract_vjepa.py` skips halves whose `.npy` already exists, so re-running is safe. Edit the slice for each chunk.

In [ ]:
from footy.config import VJEPA_GAME_PATHS_FULL, default_data_dir
import json, subprocess
import scripts.extract_vjepa as ex

WORK = "/kaggle/working/soccernet"
VIDEO_DIR = default_data_dir()
DATASET_ID = "daniels02/vjepa-features-25games"

def run_chunk(games):
    # In-process call -> live progress prints (no subprocess buffering).
    ex.run(VIDEO_DIR, WORK, batch=8, games=games)

def save_features():
    json.dump({"title":"vjepa-features-25games","id":DATASET_ID,
               "licenses":[{"name":"CC0-1.0"}]},
              open(f"{WORK}/dataset-metadata.json","w"))
    # Try create; if it fails for ANY reason (e.g. "already in use"), push a version.
    r = subprocess.run(["kaggle","datasets","create","-p",WORK,"--dir-mode","tar"],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        print("create failed -> pushing a new version instead")
        r = subprocess.run(["kaggle","datasets","version","-p",WORK,
                            "-m","update features","--dir-mode","tar"],
                           capture_output=True, text=True)
        print(r.stdout, r.stderr, "version rc:", r.returncode)

print(f"{len(VJEPA_GAME_PATHS_FULL)} games total, chunks of 5")


### Run chunk 1 (games 0-4) then save

In [ ]:
chunk = VJEPA_GAME_PATHS_FULL[0:5]
print('extracting:')
for g in chunk: print('  ', g)
run_chunk(chunk)
save_features()
print('chunk 1 done + saved')


### Run chunk 2 (games 5-9) then save

In [ ]:
chunk = VJEPA_GAME_PATHS_FULL[5:10]
print('extracting:')
for g in chunk: print('  ', g)
run_chunk(chunk)
save_features()
print('chunk 2 done + saved')


### Run chunk 3 (games 10-14) then save

In [ ]:
chunk = VJEPA_GAME_PATHS_FULL[10:15]
print('extracting:')
for g in chunk: print('  ', g)
run_chunk(chunk)
save_features()
print('chunk 3 done + saved')


### Run chunk 4 (games 15-19) then save

In [ ]:
chunk = VJEPA_GAME_PATHS_FULL[15:20]
print('extracting:')
for g in chunk: print('  ', g)
run_chunk(chunk)
save_features()
print('chunk 4 done + saved')


### Run chunk 5 (games 20-24) then save

In [ ]:
chunk = VJEPA_GAME_PATHS_FULL[20:25]
print('extracting:')
for g in chunk: print('  ', g)
run_chunk(chunk)
save_features()
print('chunk 5 done + saved')


## Final verify — all 50 feature files present

In [ ]:
import glob
n = len(glob.glob("/kaggle/working/soccernet/**/*_VJEPA21_L.npy", recursive=True))
print("feature files:", n, "/ 50")
import numpy as np
sample = glob.glob("/kaggle/working/soccernet/**/*_VJEPA21_L.npy", recursive=True)[:2]
for s in sample:
    f = np.load(s)
    print(s.split("soccernet/")[-1], f.shape, "finite", bool(np.isfinite(f).all()), "std", round(float(f.std()),3))
